# Fine-tuned Model Inference Test

Sau khi mô hình chạy xong tiến trình huấn luyện ở file `02_finetuning_qlora.ipynb` và xuất ra thư mục **Adapter (LoRA weights)**, file Notebook này sẽ giúp bạn ghép nối nó vào Base Model gốc để xem kết quả chất lượng tóm tắt.

In [1]:
import sys
import os
import torch
from peft import PeftModel

# Add project root to sys.path for module imports
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from modules.model_loader import load_model_and_tokenizer
from modules.dataset_utils import format_prompt

c:\Users\ezycloudx-admin\Downloads\qwen2.5-3b-meeting-summarization\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Step 1: Load Base Model and Tokenizer
Chúng ta cần tải bản gốc chuẩn bị trước. (Speed sẽ rất nhanh nếu `model_name` đã quét được từ trong cache/ổ cứng của bạn).

In [3]:
model_name = "Qwen/Qwen2.5-3B" # Thay bằng tên folder base model của bạn nếu cần tải nạp offline

base_model, tokenizer = load_model_and_tokenizer(
    model_name=model_name,
    use_4bit=True,
    torch_dtype="float16",
    device_map="auto"
)

Loading checkpoint shards: 100%|██████████| 2/2 [00:05<00:00,  2.77s/it]


### Step 2: Load Fine-tuned LoRA Adapter into the Base Model
Use `PeftModel` to layer the fine-tuned LoRA adapter on top of the base model.

In [4]:
lora_dir = os.path.join(PROJECT_ROOT, "models", "qwen25-3b-v1")

try:
    model = PeftModel.from_pretrained(base_model, lora_dir)
    print("Fine-tuned LoRA adapter loaded successfully!")
except Exception as e:
    print(f"Không tìm thấy thư mục LoRA tại: {lora_dir}. Quá trình train của bạn đã tạo ra file chưa?")
    raise e

c:\Users\ezycloudx-admin\Downloads\qwen2.5-3b-meeting-summarization\venv\lib\site-packages\peft\config.py:165: UserWarning: Unexpected keyword arguments ['alora_invocation_tokens', 'arrow_config', 'ensure_weight_tying', 'peft_version', 'qalora_group_size', 'target_parameters', 'use_qalora'] for class LoraConfig, these are ignored. This probably means that you're loading a configuration file that was saved using a higher version of the library and additional parameters have been introduced since. It is highly recommended to upgrade the PEFT version before continuing (e.g. by running `pip install -U peft`).
  warnings.warn(


Fine-tuned LoRA adapter loaded successfully!


### Step 3: Run a Meeting Summarization Test
Below is a sample meeting transcript to use as input for the model.

In [ ]:
# You can replace this with an actual .txt meeting file of your own
# Template rỗng đúng chuẩn training data v2 (part1)
test_meeting_text = """# **Tiêu đề**
## I. Nội dung chính

### 1. Mục tiêu cuộc họp

### 2. Các vấn đề đã thảo luận

### 3. Kết luận và quyết định

## II. Danh sách công việc cần làm

| Công việc | Người phụ trách | Hạn chót |
| :--- | :--- | :--- |
| | ||
| | ||
| | ||
| | ||
| | ||
[14:03] sẽ liên quan đến dự án K2 của nhóm chúng ta.
[14:03] Tiếp theo, giám đốc sẽ phổ biến cụ thể với mọi người.
[14:03] Cụ thể với mọi người ạ
[14:03] Cảm ơn mọi người đã tham gia
[14:03] tham gia đầy đủ cuộc họp ngày hôm nay. Ba tháng vừa qua,
[14:03] 3 tháng vừa qua thì tôi thấy là nhóm mình đã liên tục
[14:03] mình đã liên tục là không có đạt đạt ra ghét như là cái à
[14:03] Như là cái báo cáo mà...
[14:03] cái báo cáo mà mình gửi cho mọi người ở trong...
[14:03] trong file thì
[14:03] Chưởng phòng hãy cho tôi biết lý do
[14:03] Dạ thưa giám đốc, theo báo cáo mà các trưởng...
[14:03] Các trưởng nhóm gửi tới đối thủ rất mạnh, họ treo.
[14:03] họ trêu banner vị trí tốt và có chiến lược.
[14:03] và có chiến lược, họ có chiến lược đưa ra rất hiệu quả.
[14:03] Cảm ơn các bạn đã theo dõi và hẹn gặp lại.
[14:03] Tôi cảm thấy cái lý do mà các đối thủ nó...
[14:03] đối thủ nó không có thả đáng chẳng phải là trước đây mình dựng
[14:03] trước đây mình vẫn nằm rất là tốt dù là các đối thủ là
[14:04] là các đối thủ là rất là đáng gợm hay sao Thư ký em hãy
[14:04] Thư ký em hãy đọc cho các bạn nghe những vấn đề mà mình...
[14:04] những cái vấn đề mà mình tìm thấy trong bản báo cáo đi Vâng, thưa mọi người
[14:04] Vâng, thưa mọi người, sau khi đã xem qua bản báo cáo, cũng như quán truyền thông tin,
[14:04] cũng như quan sát quá trình làm việc của bộ phận. Giám đốc nhận thông tin về các bài hỏi,
[14:04] Giám đốc nhận thấy...
[14:04] Giám đốc nhận thấy là vấn đề lớn nhất dẫn đến việc không đại chiến.
[14:04] dẫn đến việc không đạt trị tiêu, đó là các chiến lược marketing đưa ra không hiểu.
[14:04] đưa ra không hiệu quả. Lý do chính là các mảng nhỏ trong chiến lược.
[14:04] nhỏ trong chiến lược đó đều không được hoàn thành tốt.
[14:04] ví dụ như nhân viên nhận mảng thiết kế banner
[14:04] thích kế banner chẳng mãn làm cho có khiến cho vì quản
[14:04] cho việc quảng cáo sản phẩm đến khách hàng bị thất bại.
[14:04] Các nhân viên thường xuyên đi trễ về sớm
[14:04] về sớm. Một số nhân viên thậm chí còn vắng mặt nhiều hơn mà
[14:04] nhiều hơn mà không có lý do chính đáng có rất nhiều khách hàng đã phải
"""

# Pass an empty string for output_text since we want the model to generate it
prompt = format_prompt(input_text=test_meeting_text, output_text="")

# Display the raw prompt that will be fed to the model
print(prompt)

Hãy cập nhật lại báo cáo cuộc họp trước đó bằng cách bổ sung thêm thông tin từ nội dung thảo luận mới dưới đây.

### Đầu vào (Báo cáo cũ & Nội dung thảo luận mới):
# Cải thiện chất lượng phục vụ tại chuỗi quán cà phê

## I. Nội dung chính

### 1. Mục tiêu cuộc họp


### 2. Các vấn đề đã thảo luận


### 3. Kết luận và quyết định


## II. Danh sách công việc cần làm

| Công việc | Người phụ trách | Hạn chót |
| :--- | :--- | :--- |
| |  |
|  |  |  |

[14:03] sẽ liên quan đến dự án K2 của nhóm chúng ta.
[14:03] Tiếp theo, giám đốc sẽ phổ biến cụ thể với mọi người.
[14:03] Cụ thể với mọi người ạ
[14:03] Cảm ơn mọi người đã tham gia
[14:03] tham gia đầy đủ cuộc họp ngày hôm nay. Ba tháng vừa qua,
[14:03] 3 tháng vừa qua thì tôi thấy là nhóm mình đã liên tục
[14:03] mình đã liên tục là không có đạt đạt ra ghét như là cái à
[14:03] Như là cái báo cáo mà...
[14:03] cái báo cáo mà mình gửi cho mọi người ở trong...
[14:03] trong file thì
[14:03] Chưởng phòng hãy cho tôi biết lý do
[14:03] Dạ thư

### Step 4: Run Inference (Generate the Summary)

In [6]:
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

input_length = inputs["input_ids"].shape[1]

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=1024,        # Maximum tokens for the generated summary
        temperature=0.3,           # Low temperature for stable, focused output
        top_p=0.9,                 # Filter unlikely tokens
        repetition_penalty=1.1,    # Penalise repetition
        pad_token_id=tokenizer.pad_token_id,
    )

# Strip the prompt prefix — keep only the newly generated text
generated_tokens = outputs[0, input_length:]
result = tokenizer.decode(generated_tokens, skip_special_tokens=True)

print("====== AI-GENERATED SUMMARY ======")
print(result)

c:\Users\ezycloudx-admin\Downloads\qwen2.5-3b-meeting-summarization\venv\lib\site-packages\transformers\generation\configuration_utils.py:590: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.3` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
c:\Users\ezycloudx-admin\Downloads\qwen2.5-3b-meeting-summarization\venv\lib\site-packages\transformers\generation\configuration_utils.py:595: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.9` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(


====== AI-GENERATED SUMMARY ======
# Cải thiện chất lượng phục vụ tại chuỗi quán cà phê

## I. Nội dung chính

### 1. Mục tiêu cuộc họp
- Đánh giá tình hình chất lượng dịch vụ hiện tại.
- Giải quyết các tồn đọng về thái độ phục vụ và tốc độ phục vụ.

### 2. Các vấn đề đã thảo luận
- Thái độ phục vụ của một số nhân viên chưa đạt yêu cầu.
- Tốc độ phục vụ giảm sút so với quy chuẩn.

### 3. Kết luận và quyết định
- Triển khai đào tạo lại toàn bộ nhân viên về礼仪和服务态度.
- Áp dụng biểu đồ thời gian để quản lý tiến độ phục vụ.

## II. Danh sách công việc cần làm

| Công việc | Người phụ trách | Hạn chót |
| :--- | :--- | :--- |
| Tổ chức đào tạo礼仪 cho toàn bộ nhân viên | Quản lý chuỗi | 15/06/2026 |
| Lập bảng biểu đồ thời gian theo dõi tiến độ | Phòng Nhân sự | 18/06/2026 |


### Step 5: Evaluate on Test Set (ROUGE + BERTScore)

Các cell dưới đây sẽ:
- Đọc test set từ thư mục dữ liệu gốc.
- Sinh summary bằng model fine-tuned.
- Tính ROUGE và BERTScore cho tiếng Việt.

In [4]:
# Nếu thiếu thư viện, bỏ comment để cài
# %pip install -q evaluate rouge_score bert-score

import re
import glob
import numpy as np
import evaluate
from tqdm.auto import tqdm

from modules.dataset_utils import parse_meeting_file, format_prompt


def clean_generated_text(text: str) -> str:
    text = re.sub(r"\s+", " ", text).strip()
    return text


def generate_summary(model, tokenizer, input_text: str, max_new_tokens: int = 512) -> str:
    prompt = format_prompt(input_text=input_text, output_text="")
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    input_length = inputs["input_ids"].shape[1]

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.2,
            top_p=0.9,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.pad_token_id,
        )

    gen_tokens = outputs[0, input_length:]
    pred = tokenizer.decode(gen_tokens, skip_special_tokens=True)
    return clean_generated_text(pred)


def load_test_samples(data_dir: str, limit: int | None = 30):
    files = sorted(glob.glob(os.path.join(data_dir, "*.txt")))
    samples = []

    for fp in files:
        src, ref = parse_meeting_file(fp)
        if src and ref:
            samples.append({"file": fp, "input": src, "reference": ref.strip()})

    if limit is not None:
        samples = samples[:limit]

    return samples

In [8]:
# Chọn đúng thư mục test data của bạn
TEST_DATA_DIR = os.path.join(PROJECT_ROOT, "data", "raw", "v3")
MAX_TEST_SAMPLES = 20  # tăng nếu muốn đánh giá đầy đủ hơn

samples = load_test_samples(TEST_DATA_DIR, limit=MAX_TEST_SAMPLES)
print(f"Loaded {len(samples)} test samples from: {TEST_DATA_DIR}")

predictions = []
references = []

for item in tqdm(samples, desc="Generating summaries"):
    pred = generate_summary(model, tokenizer, item["input"], max_new_tokens=512)
    predictions.append(pred)
    references.append(item["reference"])

print("Done generating predictions.")
print("Example prediction:\n", predictions[0][:500] if predictions else "N/A")

Loaded 20 test samples from: c:\Users\ezycloudx-admin\Downloads\qwen2.5-3b-meeting-summarization\data\raw\v3


Generating summaries: 100%|██████████| 20/20 [13:45<00:00, 41.26s/it]

Done generating predictions.
Example prediction:
 # Chiến lược ra mắt sản phẩm gia dụng thông minh và chiến thuật cạnh tranh ## I. Nội dung chính ### 1. Mục tiêu cuộc họp - Định hướng chiến lược bán hàng cho dòng sản phẩm gia dụng thông minh mới. - Thảo luận phương án ứng phó với áp lực cạnh tranh từ các thương hiệu khác về giá. ### 2. Các vấn đề đã thảo luận - Thị trường: Sức mua hồi phục, khách hàng ưu tiên tính năng tiết kiệm điện và thiết kế. - Đối thủ: Thương hiệu X đang cạnh tranh bằng các chương trình giảm giá sâu (20%). ### 3. Kết luận 


In [9]:
rouge = evaluate.load("rouge")
bertscore = evaluate.load("bertscore")

rouge_result = rouge.compute(
    predictions=predictions,
    references=references,
    use_stemmer=False,
)

bertscore_result = bertscore.compute(
    predictions=predictions,
    references=references,
    lang="vi",
    model_type="xlm-roberta-large",
)

metrics = {
    "rouge1": rouge_result.get("rouge1"),
    "rouge2": rouge_result.get("rouge2"),
    "rougeL": rouge_result.get("rougeL"),
    "bertscore_precision": float(np.mean(bertscore_result["precision"])) if bertscore_result.get("precision") else None,
    "bertscore_recall": float(np.mean(bertscore_result["recall"])) if bertscore_result.get("recall") else None,
    "bertscore_f1": float(np.mean(bertscore_result["f1"])) if bertscore_result.get("f1") else None,
    "num_samples": len(predictions),
}

print("===== TEST SET METRICS =====")
for k, v in metrics.items():
    if isinstance(v, float):
        print(f"{k}: {v:.4f}")
    else:
        print(f"{k}: {v}")

c:\Users\ezycloudx-admin\Downloads\qwen2.5-3b-meeting-summarization\venv\lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\ezycloudx-admin\.cache\huggingface\hub\models--xlm-roberta-large. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 391/391 [00:00<00:00, 4159.00it/s]


===== TEST SET METRICS =====
rouge1: 0.8413
rouge2: 0.6758
rougeL: 0.6671
bertscore_precision: 0.9403
bertscore_recall: 0.9524
bertscore_f1: 0.9463
num_samples: 20
